##### Copyright 2019 The TensorFlow Authors.

# Deep Learning · Lab 11

## 2. Text generation with a GRU

This notebook is based on the [TensorFlow text generation tutorial](https://www.tensorflow.org/text/tutorials/text_generation). Use this distributed copy for the lab. This edition uses the English Shakespeare corpus from that tutorial.

### T00 — Check TensorFlow, Keras and GPU

**First executable cell.** Select a GPU and runtime version **2026.04** in Colab before running any code. This setup initializes the legacy Keras API used by the original model.

In [ ]:
# T00 — Colab setup for the original legacy Keras model.
import os
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

try:
    tf_version = version("tensorflow")
except PackageNotFoundError:
    tf_version = ""
if not tf_version.startswith("2.19."):
    raise RuntimeError("Colab: choose GPU and runtime version 2026.04 (TensorFlow 2.19), then rerun T00.")

try:
    legacy_version = version("tf-keras")
except PackageNotFoundError:
    legacy_version = None

if legacy_version != "2.19.0":
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "--no-deps", "tf-keras==2.19.0"
    ])

os.environ["TF_USE_LEGACY_KERAS"] = "1"
import tensorflow as tf

keras_backend = tf.keras.__name__
gpu_devices = tf.config.list_physical_devices("GPU")
print("TensorFlow:", tf.__version__)
print("Keras:", keras_backend)
print("GPU:", gpu_devices)
if not keras_backend.startswith("tf_keras"):
    raise RuntimeError("Legacy Keras was not loaded. Restart the Colab session and run T00 first.")
if not gpu_devices:
    raise RuntimeError("Colab: Runtime > Change runtime type > GPU > Save. Then rerun T00.")


In [ ]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

The model learns to predict the next character in the English Shakespeare corpus. Feeding its predictions back into the model lets it produce new text.

In T36, the generator starts with `ROMEO:` and adds 1,000 characters. Inspect your own sample for speaker labels, line breaks and word-like fragments. Samples vary between runs.

The model can pick up patterns of spelling and punctuation. Recognizable fragments may appear alongside incomplete sentences or misspelled words.

## Setup

### T01 — Import TensorFlow and helper libraries

In [ ]:
import tensorflow as tf

import numpy as np
import os
import time

import os
os.environ['TF_USE_LEGACY_KERAS']='1'

The lab downloads the English Shakespeare corpus as `shakespeare.txt`.

### T02 — Download the corpus

In [ ]:
path_to_file = tf.keras.utils.get_file('shakespeare.txt', 'https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt')

### T03 — Decode the text as UTF-8

In [ ]:
# Read, then decode for py2 compat.
text = open(path_to_file, 'rb').read().decode(encoding='utf-8')
# length of text is the number of characters in it
print(f'Length of text: {len(text)} characters')

### T04 — Display the first 250 characters

In [ ]:
# Take a look at the first 250 characters in text
print(text[:250])

### T05 — List the distinct characters

In [ ]:
# The unique characters in the file
vocab = sorted(set(text))
print(f'{len(vocab)} unique characters')

## From corpus to training pairs

### T06 — Map between characters and indices

In [ ]:
char2idx = {u:i for i, u in enumerate(vocab)}
idx2char = np.array(vocab)

text_as_int = np.array([char2idx[c] for c in text]) #szöveget pontosabban a karaktereket számokká konvertáljuk

### T07 — Inspect the index mapping

In [ ]:
print('{')
for char,_ in zip(char2idx, range(20)):
    print('  {:4s}: {:3d},'.format(repr(char), char2idx[char]))
print('  ...\n}')

### T08 — Encode the first 13 characters

In [ ]:
print ('{} ---- characters mapped to int ---- > {}'.format(repr(text[:13]), text_as_int[:13]))

At each time step the target is the next character. Inputs and targets are sequences of integer indices.

For the chunk `Hello`, for example, the input is `Hell` and the target is `ello`.

### T09 — Build a stream of single characters

In [ ]:
seq_length = 100
examples_per_epoch = len(text)//(seq_length+1)


char_dataset = tf.data.Dataset.from_tensor_slices(text_as_int)

for i in char_dataset.take(10):
  print(idx2char[i.numpy()])

### T10 — Build chunks of 101 characters

In [ ]:
sequences = char_dataset.batch(seq_length+1, drop_remainder=True)

for item in sequences.take(5):
  print(repr(''.join(idx2char[item.numpy()])))

### T11 — Split the input and target

In [ ]:
def split_input_target(chunk):
    input_text = chunk[:-1]
    target_text = chunk[1:]
    return input_text, target_text



### T12 — Try the one-character shift

In [ ]:
split_input_target(list("Tensorflow"))

### T13 — Build training pairs for the whole dataset

In [ ]:
dataset = sequences.map(split_input_target)

### T14 — Inspect one input–target pair

In [ ]:
for input_example, target_example in  dataset.take(1):
  print ('Input data: ', repr(''.join(idx2char[input_example.numpy()])))
  print ('Target data:', repr(''.join(idx2char[target_example.numpy()])))

Shuffle the pairs and group them in batches of 64. The order of characters within each sequence stays intact.

### T15 — Match the first five time steps

In [ ]:
for i, (input_idx, target_idx) in enumerate(zip(input_example[:5], target_example[:5])):
    print("Step {:4d}".format(i))
    print("  input: {} ({:s})".format(input_idx, repr(idx2char[input_idx])))
    print("  expected output: {} ({:s})".format(target_idx, repr(idx2char[target_idx])))

### T16 — Shuffle and group 64 training pairs

In [ ]:
BATCH_SIZE = 64
BUFFER_SIZE = 10000
dataset = dataset.shuffle(BUFFER_SIZE).batch(BATCH_SIZE, drop_remainder=True)

dataset

## Build the model

The `tf.keras.Sequential` model has three layers: Embedding, GRU and Dense. At each time step the final layer produces one raw score (logit) per vocabulary character.

### T17 — Set three model dimensions

In [ ]:
# Length of the vocabulary in StringLookup Layer
vocab_size = len(vocab)

# The embedding dimension
embedding_dim = 256

# Number of RNN units
rnn_units = 1024

### T18 — Define the Embedding, GRU and Dense model

In [ ]:
def build_model(vocab_size, embedding_dim, rnn_units, batch_size):

    model = tf.keras.Sequential([
    tf.keras.layers.Embedding(vocab_size, embedding_dim,batch_input_shape=[batch_size, None]),
    tf.keras.layers.GRU(rnn_units,return_sequences=True,stateful=True,recurrent_initializer='glorot_uniform'),
    tf.keras.layers.Dense(vocab_size)
  ])
    return model

#Ez a kaggleről van

### T19 — Build the training model

In [ ]:
model = build_model(vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    rnn_units=rnn_units, batch_size=BATCH_SIZE)

### T20 — Read the model summary

In [ ]:
model.summary()

The GRU stores recurrent state for each batch position, so the batch size is fixed. Because the training sequences were shuffled, state carried across batches does not follow the book's original sequence.

## Try the untrained model

### T21 — Try a forward pass on one batch

In [ ]:
for input_example_batch, target_example_batch in dataset.take(1):
  example_batch_predictions = model(input_example_batch)
  print(example_batch_predictions.shape, "# (batch_size, sequence_length, vocab_size)")

### T22 — Read the model summary again

In [ ]:
model.summary()

`tf.random.categorical` samples using logits. Higher-scoring options are more likely, but the top-scoring option is not necessarily selected on every run.

### T23 — Sample from the trial output

In [ ]:
sampled_indices = tf.random.categorical(example_batch_predictions[0], num_samples=1)
sampled_indices = tf.squeeze(sampled_indices,axis=-1).numpy()

### T24 — Inspect the sampled indices

In [ ]:
sampled_indices

### T25 — Decode the untrained model's sample

In [ ]:
print("Input: \n", repr("".join(idx2char[input_example_batch[0]])))
print()
print("Next Char Predictions: \n", repr("".join(idx2char[sampled_indices ])))

## Training and checkpoints

At each time step, predict which character class is correct next.

### T26 — Set up the loss function

In [ ]:
loss = tf.losses.SparseCategoricalCrossentropy(from_logits=True)

### T27 — Inspect the initial mean loss

In [ ]:
example_batch_mean_loss = loss(target_example_batch, example_batch_predictions)
print("Prediction shape: ", example_batch_predictions.shape, " # (batch_size, sequence_length, vocab_size)")
print("Mean loss:        ", example_batch_mean_loss)

At initialization, when predictions are roughly uniform, the exponential of the mean loss is close to the vocabulary size.

### T28 — Exponentiate the mean loss

In [ ]:
tf.exp(example_batch_mean_loss).numpy()

### T29 — Compile the training model

In [ ]:
model.compile(optimizer='adam', loss=loss)

### Saving weights

### T30 — Configure weight checkpoints

In [ ]:
# Directory where the checkpoints will be saved
checkpoint_dir = './training_checkpoints'
# Name of the checkpoint files
checkpoint_prefix = os.path.join(checkpoint_dir, "ckpt_{epoch}")

checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_prefix,
    save_weights_only=True)

The original code trains for **30 epochs**. Keep the GPU session running until training completes.

### T31 — Set the epoch count

In [ ]:
EPOCHS = 30

### T32 — Train the character model

In [ ]:
history = model.fit(dataset, epochs=EPOCHS, callbacks=[checkpoint_callback])

### T33 — Find the latest checkpoint

In [ ]:
tf.train.latest_checkpoint(checkpoint_dir)

For generation, rebuild the model with the same layers at batch size one and load the weights from a local checkpoint.

### T34 — Rebuild with batch size one and restore the weights

In [ ]:
model = build_model(vocab_size, embedding_dim, rnn_units, batch_size=1)

model.load_weights(tf.train.latest_checkpoint(checkpoint_dir))

model.build(tf.TensorShape([1, None]))

## Generate text by sampling

After processing the prefix, the function samples a character and uses its index as the input to the next prediction.

### T35 — Define the generator

In [ ]:
def generate_text(model, start_string):

    num_generate = 1000
    input_eval = [char2idx[s] for s in start_string]
    input_eval = tf.expand_dims(input_eval, 0)


    text_generated = []

    temperature = 1.0

    model.reset_states()
    for i in range(num_generate):
        predictions = model(input_eval)
        predictions = tf.squeeze(predictions, 0) #1-es dimenziókat eltávolítja
        predictions = predictions / temperature
        predicted_id = tf.random.categorical(predictions, num_samples=1)[-1,0].numpy()

        input_eval = tf.expand_dims([predicted_id], 0)

        text_generated.append(idx2char[predicted_id])

    return (start_string + ''.join(text_generated))

### T36 — Generate text from the ROMEO: prefix

In [ ]:
print(generate_text(model, start_string=u"ROMEO:"))

The remaining cells show the shapes in one step. Recurrent state can change, but the trained weights stay the same.

### T37 — Inspect one generation step

In [ ]:
text =[]
input_eval = [char2idx[s] for s in 'A']
input_eval = tf.expand_dims(input_eval, 0)
predictions = model(input_eval)

### T38 — Remove the one-item batch axis

In [ ]:
predictions = tf.squeeze(predictions, 0)

### T39 — Inspect a single character score

In [ ]:
predictions[0,1]

### T40 — Sample one character index

In [ ]:
predicted_id = tf.random.categorical(predictions, num_samples=1)[-1,0].numpy()

### T41 — Build the next input and collect the character

In [ ]:
input_eval = tf.expand_dims([predicted_id], 0)
text.append(idx2char[predicted_id])